In [4]:
import os
import requests

In [46]:
base_url = "https://api.panoramax.xyz"
picture_id ="6058367b-b429-4f34-8955-674ace44c18b"

output_dir = "streetview_images"
output_filename = f"{picture_id}.jpg"
output_path = os.path.join(output_dir, output_filename)

def download_image(picture_id):
    try:
        path = f"api/pictures/{picture_id}"
        metadata_url = f"{base_url}/{path}"
        os.makedirs(output_dir, exist_ok=True)
        print(f"Directory verified: {output_dir}")

        # Get the picture metadata (JSON response)
        print(f"Fetching metadata from: {metadata_url}")
        response = requests.get(metadata_url)
        response.raise_for_status() # Check for HTTP errors
        data = response.json()

        # extract azimuth
        properties = data.get('properties', {})

        # Extract the image URL from the response
        # Panoramax/GeoVisio returns a STAC Item. The image links are in the 'assets' dictionary.
        # We prioritize 'hd' (High Definition) but can fall back to 'sd' (Standard Definition).
        assets = data.get('assets', {})
        
        if 'hd' in assets:
            image_url = assets['hd']['href']
            print("Found HD image URL.")
        elif 'sd' in assets:
            image_url = assets['sd']['href']
            print("HD not available, found SD image URL.")
        else:
            raise ValueError("No suitable image asset ('hd' or 'sd') found in the response.")

        print(f"Image URL extracted: {image_url}")
        print("Downloading image...")
        image_response = requests.get(image_url)
        image_response.raise_for_status()
        with open(output_path, 'wb') as file:
            file.write(image_response.content)
        print(f"Success! Image saved to: {output_path}")
        return data

    except requests.exceptions.RequestException as e:
        print(f"Network error occurred: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")
    

In [6]:
import requests
import math
import os

# --- Configuration ---
BASE_URL = "https://api.panoramax.xyz"
SEARCH_URL = f"{BASE_URL}/api/search"

# Center Point lat:  Center Point lon: 
LATITUDE = 49.17359902913684
LONGITUDE = -0.3482916490346519
SEARCH_RADIUS_METERS = 5 
OUTPUT_DIR = "streetview_360"

def get_bbox_from_point(lat, lon, dist_meters):
    earth_radius = 6378137
    d_lat = dist_meters / earth_radius
    d_lon = dist_meters / (earth_radius * math.cos(math.pi * lat / 180))
    
    # Offsets in degrees
    d_lat_deg = d_lat * 180 / math.pi
    d_lon_deg = d_lon * 180 / math.pi
    
    return f"{lon - d_lon_deg},{lat - d_lat_deg},{lon + d_lon_deg},{lat + d_lat_deg}"

In [47]:
data = download_image("0e4308df-0f5e-45e2-84ef-4b350a7b33cb")

Directory verified: streetview_images
Fetching metadata from: https://api.panoramax.xyz/api/pictures/0e4308df-0f5e-45e2-84ef-4b350a7b33cb
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/0e/43/08/df/0f5e-45e2-84ef-4b350a7b33cb.jpg
Success! Image saved to: streetview_images/6058367b-b429-4f34-8955-674ace44c18b.jpg


In [48]:
data

{'id': '0e4308df-0f5e-45e2-84ef-4b350a7b33cb',
 'bbox': [-0.407263735, 49.189170069, -0.407263735, 49.189170069],
 'type': 'Feature',
 'links': [{'rel': 'root',
   'href': 'https://api.panoramax.xyz/api/',
   'type': 'application/json',
   'title': 'Instance catalog'},
  {'rel': 'parent',
   'href': 'https://api.panoramax.xyz/api/collections/a5ebcf2e-0235-46f9-b7d3-634cf4eeab85',
   'type': 'application/json'},
  {'rel': 'self',
   'href': 'https://api.panoramax.xyz/api/collections/a5ebcf2e-0235-46f9-b7d3-634cf4eeab85/items/0e4308df-0f5e-45e2-84ef-4b350a7b33cb',
   'type': 'application/geo+json'},
  {'rel': 'collection',
   'href': 'https://api.panoramax.xyz/api/collections/a5ebcf2e-0235-46f9-b7d3-634cf4eeab85',
   'type': 'application/json'},
  {'rel': 'license',
   'href': 'https://creativecommons.org/licenses/by-sa/4.0/',
   'title': 'License for this object (CC-BY-SA-4.0)'},
  {'id': 'fba870e1-25aa-4082-a4fb-56a74768dee5',
   'rel': 'next',
   'href': 'https://api.panoramax.xyz/api

In [36]:
fov = properties.get('pers:interior_orientation').get('field_of_view')

In [38]:
if fov and fov >= 360: print("true")

true


In [52]:
azimuth = properties.get('view:azimuth')
azimuth

41

In [9]:
# --- Helper: Calculate Distance (Haversine Formula) ---
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the great-circle distance between two points in meters.
    """
    R = 6378137  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [39]:
def is_360_panorama(feature):
    """
    Determines if a STAC feature is a 360 degree panorama using multiple checks.
    """
    props = feature.get("properties", {})
    assets = feature.get("assets", {})
    
    # Check 1: Explicit Projection Type (The Standard)
    if props.get("GPano:ProjectionType") == "equirectangular":
        return True

    # Check 2: Explicit Field of View
    # Some uploads might lack GPano tags but have STAC view properties
    
    fov = props.get('pers:interior_orientation').get("field_of_view")
    if fov and fov >= 360:
        return True

    # Check 3: Aspect Ratio Heuristic (The Fallback)
    # If metadata is missing, we check if the image dimensions are 2:1
    # We look for dimensions in 'hd' or 'sd' assets, or properties
    width = props.get("exif:pixelXDimension")
    height = props.get("exif:pixelYDimension")

    # If not in properties, try to find it in assets (sometimes hidden there)
    if not width or not height:
        for key in ['hd', 'sd', 'visual']:
            asset = assets.get(key, {})
            # Some STAC items put dims in the asset metadata
            if 'proj:shape' in asset: # [height, width]
                h, w = asset['proj:shape']
                if w > 0 and h > 0:
                    ratio = w / h
                    # Allow small margin of error for compression artifacts (1.9 to 2.1)
                    if 1.9 < ratio < 2.1:
                        return True

    return False

In [55]:
# function that returns all images in search radius
# the images are are here the features
# 'properties' contains metadate
# 'assets' contains the image url
# 'geometry' contains infromation about the location where the image was taken
def get_images_at(lat,lon,rad):
  bbox_string = get_bbox_from_point(lat,lon,rad)
  print(f"Searching area: {bbox_string}")

  params = {
    "bbox": bbox_string,
    "limit": 100
  }

  try:
    # request
    response = requests.get(SEARCH_URL, params=params)
    response.raise_for_status()
    features = response.json().get("features", [])
    
    print(f"Total images found: {len(features)}")
    
    def get_distance(x):
       geo = x.get('geometry',{}).get("coordinates", [])
       if len(geo) > 2:
          # images where distance can not be calculated should be at the end
          return math.inf
       img_lon, img_lat = geo[0], geo[1]
       dist = haversine_distance(lat, lon, img_lat, img_lon)
       # save calculated distance for later uses
       x["distance_from_location"] = dist
       return dist


    filtered_features = sorted(
      filter(is_360_panorama,features),
      key=get_distance)
    # for feature in features:
    #     props = feature.get("properties", {})
    #     projection = props.get("GPano:ProjectionType")      
    #     if projection == "equirectangular":
    #         filtered_features.append(feature)
    print(f"360° images found: {len(filtered_features)}")
    return filtered_features

  except Exception as e:
    print(f"Error: {e}")
    return []
  


In [ ]:
LATITUDE, LONGITUDE = 49.1891701, -0.4072637
RADIUS = 3.0

some_images = get_images_at(LATITUDE,LONGITUDE,RADIUS)

Searching area: -0.4073049346332098,49.189143150541476,-0.4072224653667902,49.18919704945852
Total images found: 6
360° images found: 4


[{'id': '0e4308df-0f5e-45e2-84ef-4b350a7b33cb',
  'bbox': [-0.407263735, 49.189170069, -0.407263735, 49.189170069],
  'type': 'Feature',
  'links': [{'rel': 'root',
    'href': 'https://api.panoramax.xyz/api/',
    'type': 'application/json',
    'title': 'Instance catalog'},
   {'rel': 'parent',
    'href': 'https://api.panoramax.xyz/api/collections/a5ebcf2e-0235-46f9-b7d3-634cf4eeab85',
    'type': 'application/json'},
   {'rel': 'self',
    'href': 'https://api.panoramax.xyz/api/collections/a5ebcf2e-0235-46f9-b7d3-634cf4eeab85/items/0e4308df-0f5e-45e2-84ef-4b350a7b33cb',
    'type': 'application/geo+json'},
   {'rel': 'collection',
    'href': 'https://api.panoramax.xyz/api/collections/a5ebcf2e-0235-46f9-b7d3-634cf4eeab85',
    'type': 'application/json'},
   {'rel': 'license',
    'href': 'https://creativecommons.org/licenses/by-sa/4.0/',
    'title': 'License for this object (CC-BY-SA-4.0)'},
   {'id': 'fba870e1-25aa-4082-a4fb-56a74768dee5',
    'rel': 'next',
    'href': 'https:/

In [56]:

def down_load_images_from_features(features,out_dir="streetview_images"):
    """
    Loads images from image_url property and saves them in the out_dir
    path is now saved in feature.path"""
    for feature in features:
        picture_id = feature.get('id')
        output_filename = f"{picture_id}.jpg"
        output_path = os.path.join(output_dir, output_filename)

        assets = feature.get('assets', {})
        if 'hd' in assets:
            image_url = assets['hd']['href']
            print("Found HD image URL.")
        elif 'sd' in assets:
            image_url = assets['sd']['href']
            print("HD not available, found SD image URL.")
        else:
            raise ValueError("No suitable image asset ('hd' or 'sd') found in the response.")

        print(f"Image URL extracted: {image_url}")
        print("Downloading image...")
        image_response = requests.get(image_url)
        image_response.raise_for_status()
        with open(output_path, 'wb') as file:
            file.write(image_response.content)
        print(f"Success! Image saved to: {output_path}")
        # output_path saved to features
        feature['path'] = output_path
    return features

In [50]:
some_images = get_images_at(LATITUDE,LONGITUDE,RADIUS)
some_images = down_load_images_from_features(some_images)

Searching area: -0.4073049346332098,49.189143150541476,-0.4072224653667902,49.18919704945852
Total images found: 6
360° images found: 4
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/0e/43/08/df/0f5e-45e2-84ef-4b350a7b33cb.jpg
Success! Image saved to: streetview_images/0e4308df-0f5e-45e2-84ef-4b350a7b33cb.jpg
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/08/e5/c5/78/7259-4f10-a4ba-59c65d198085.jpg
Success! Image saved to: streetview_images/08e5c578-7259-4f10-a4ba-59c65d198085.jpg
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/fb/a8/70/e1/25aa-4082-a4fb-56a74768dee5.jpg
Success! Image saved to: streetview_images/fba870e1-25aa-4082-a4fb-56a74768dee5.jpg
Found HD image URL.
Image URL extracted: https://panoramax.openstreetmap.fr/images/f0/49/50/86/bed8-4655-95f6-e7e2ed489335.jpg
Success! Image saved to: streetview_images/f0495086-bed8-4655-95f6-e7e2ed489335.jpg


In [54]:
[x.get('_distance_from_query') for x in some_images]

[0.0042886953916378805,
 2.9646146543243055,
 3.0786117282522234,
 3.6938339183800837]